# Байесовская сеть: метеорит
Узлы (у всех состояния no / yes):
- **A** — метеорит на курсе столкновения (учёные уверены, что попадёт), P = 0.999
- **B** — эвакуация в бункер спасает людей, P(B | A) = 0.3
- **C** — метеорит взорвали, P(C | A) = 0.01
- **D** — после взрыва остались большие обломки, P(D | C) = 0.2
- **Impact** — до Земли долетает метеорит или крупные обломки: A и (не C или D)
- **X** — человечество уничтожено: удар и бункер не спас

Допущения: если метеорит не летит в Землю (A = no), его не взрывают и не эвакуируются; без взрыва нет обломков.
Impact и X — детерминированные узлы (вероятности 0 или 1).

In [ ]:
!pip install -q pgmpy

In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
    ('A', 'B'), ('A', 'C'), ('C', 'D'),
    ('A', 'Impact'), ('C', 'Impact'), ('D', 'Impact'),
    ('Impact', 'X'), ('B', 'X'),
])
s = ['no', 'yes']

cpd_A = TabularCPD('A', 2, [[0.001], [0.999]], state_names={'A': s})

cpd_B = TabularCPD('B', 2,              # столбцы: A=no, A=yes
    [[1.0, 0.7],                         # бункер НЕ спас
     [0.0, 0.3]],                        # бункер спас
    evidence=['A'], evidence_card=[2], state_names={'B': s, 'A': s})

cpd_C = TabularCPD('C', 2,
    [[1.0, 0.99],
     [0.0, 0.01]],
    evidence=['A'], evidence_card=[2], state_names={'C': s, 'A': s})

cpd_D = TabularCPD('D', 2,              # столбцы: C=no, C=yes
    [[1.0, 0.8],
     [0.0, 0.2]],
    evidence=['C'], evidence_card=[2], state_names={'D': s, 'C': s})

# Impact = A and (not C or D); столбцы перебирают (A, C, D): 000, 001, ..., 111
impact_yes = [1.0 if a and (not c or d) else 0.0
              for a in (0, 1) for c in (0, 1) for d in (0, 1)]
cpd_Impact = TabularCPD('Impact', 2, [[1 - v for v in impact_yes], impact_yes],
    evidence=['A', 'C', 'D'], evidence_card=[2, 2, 2],
    state_names={'Impact': s, 'A': s, 'C': s, 'D': s})

# X = Impact and not B; столбцы (Impact, B): 00, 01, 10, 11
cpd_X = TabularCPD('X', 2,
    [[1.0, 1.0, 0.0, 1.0],
     [0.0, 0.0, 1.0, 0.0]],
    evidence=['Impact', 'B'], evidence_card=[2, 2],
    state_names={'X': s, 'Impact': s, 'B': s})

model.add_cpds(cpd_A, cpd_B, cpd_C, cpd_D, cpd_Impact, cpd_X)
assert model.check_model()
for c in model.get_cpds():
    print(c)

In [ ]:
infer = VariableElimination(model)
marg = {v: infer.query([v], show_progress=False) for v in model.nodes()}
for v, f in marg.items():
    print(v, {st: round(float(p), 7) for st, p in zip(f.state_names[v], f.values)})

print('\nP(уничтожения) =', round(float(marg['X'].values[1]), 7))
print('Проверка вручную: 0.999 * 0.7 * (0.99 + 0.01*0.2) =', 0.999 * 0.7 * (0.99 + 0.01 * 0.2))

## Визуализация (как в UnBBayes)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle

pos = {'A': (5, 9), 'B': (1.2, 6), 'C': (8.8, 6), 'D': (8.8, 3),
       'Impact': (5, 3), 'X': (3, 0.3)}
W, H = 3.2, 1.9

def draw(marginals, title='Априорные вероятности'):
    fig, ax = plt.subplots(figsize=(11, 9))
    ax.set_xlim(-1, 11); ax.set_ylim(-1, 10.5); ax.axis('off'); ax.set_title(title)
    for u, v in model.edges():
        (x1, y1), (x2, y2) = pos[u], pos[v]
        if abs(y1 - y2) < 0.1:   # узлы на одной высоте: стрелка между боковыми гранями
            sgn = 1 if x2 > x1 else -1
            a, b = (x1 + sgn * W/2, y1), (x2 - sgn * W/2, y2)
        else:
            a, b = (x1, y1 - H/2), (x2, y2 + H/2)
        ax.add_patch(FancyArrowPatch(a, b, arrowstyle='-|>', mutation_scale=18, color='#1f3c88'))
    for n, (x, y) in pos.items():
        f = marginals[n]; states = f.state_names[n]
        ax.add_patch(Rectangle((x - W/2, y - H/2), W, H, fc='#fff36b', ec='#1f3c88', lw=1.5))
        ax.text(x, y + H/2 - 0.3, n, ha='center', va='center', fontsize=12, weight='bold')
        for i, (st, p) in enumerate(zip(states, f.values)):
            yy = y + 0.05 - i * 0.6
            ax.text(x - W/2 + 0.1, yy, st, va='center', fontsize=10)
            ax.text(x + 0.45, yy, f'{p*100:.2f}%', va='center', ha='right', fontsize=10)
            ax.add_patch(Rectangle((x + 0.55, yy - 0.2), 0.95 * p, 0.4, fc='#3b5bdb'))
            ax.add_patch(Rectangle((x + 0.55, yy - 0.2), 0.95, 0.4, fill=False, ec='gray', lw=0.6))
    plt.show()

draw(marg)

## Пример вывода по свидетельству
Если человечество уничтожено — какова вероятность, что метеорит взорвали? А если метеорит взорвали — каков риск?

In [ ]:
print(infer.query(['C'], evidence={'X': 'yes'}, show_progress=False))
print(infer.query(['X'], evidence={'C': 'yes'}, show_progress=False))